In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [5]:
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model="gpt-5-nano",
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [7]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.")],
        "email": "سلام سینا، فردا برای جلسه‌مان دیر می‌رسم. می‌توانیم وقت دیگری بگذاریم؟ با احترام، علیرضا."
    },
    config=config
)

In [9]:
from pprint import pprint

pprint(response)

{'__interrupt__': [Interrupt(value={'action_requests': [{'args': {'body': 'سلام '
                                                                          'علیرضا '
                                                                          'جان، '
                                                                          'ممنون '
                                                                          'از '
                                                                          'اطلاع.\n'
                                                                          'بله، '
                                                                          'حتما '
                                                                          'می\u200cتونیم '
                                                                          'زمان '
                                                                          'جلسه '
                                                                          'را '
     

In [11]:
print(response['__interrupt__'])

[Interrupt(value={'action_requests': [{'name': 'send_email', 'args': {'body': 'سلام علیرضا جان، ممنون از اطلاع.\nبله، حتما می\u200cتونیم زمان جلسه را تغییر بدیم. چه زمانی برایت مناسب است؟ اگر دوست داری، چند گزینه پیشنهاد می\u200cکنم:\n- فردا ظهر (12:00–13:00)\n- فردا بعد از ظهر (14:30–16:00)\n- روز دوشنبه صبح (10:00–12:00)\nاگر هیچ\u200cکدام مناسب نیست، لطفاً زمان\u200cهای دلخواهت را بگو تا هماهنگ کنم.\n\nبا احترام،\nسینا'}, 'description': "Tool execution requires approval\n\nTool: send_email\nArgs: {'body': 'سلام علیرضا جان، ممنون از اطلاع.\\nبله، حتما می\\u200cتونیم زمان جلسه را تغییر بدیم. چه زمانی برایت مناسب است؟ اگر دوست داری، چند گزینه پیشنهاد می\\u200cکنم:\\n- فردا ظهر (12:00–13:00)\\n- فردا بعد از ظهر (14:30–16:00)\\n- روز دوشنبه صبح (10:00–12:00)\\nاگر هیچ\\u200cکدام مناسب نیست، لطفاً زمان\\u200cهای دلخواهت را بگو تا هماهنگ کنم.\\n\\nبا احترام،\\nسینا'}"}], 'review_configs': [{'action_name': 'send_email', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}]}, id='3

In [13]:
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

سلام علیرضا جان، ممنون از اطلاع.
بله، حتما می‌تونیم زمان جلسه را تغییر بدیم. چه زمانی برایت مناسب است؟ اگر دوست داری، چند گزینه پیشنهاد می‌کنم:
- فردا ظهر (12:00–13:00)
- فردا بعد از ظهر (14:30–16:00)
- روز دوشنبه صبح (10:00–12:00)
اگر هیچ‌کدام مناسب نیست، لطفاً زمان‌های دلخواهت را بگو تا هماهنگ کنم.

با احترام،
سینا


## تأیید

In [15]:
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='a4ddfffc-5068-4d58-a216-965844f6f3e0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 166, 'total_tokens': 569, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dqf5EBY7KKMoxL3xrcmXZqujAArll', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec64c-5c93-7890-8e5e-8085f1face2b-0', tool_calls=[{'name': 're

## رد کردن

In [17]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "نه، لطفاً امضا کن - بگو دوست مهربانت سینا."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='a4ddfffc-5068-4d58-a216-965844f6f3e0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 166, 'total_tokens': 569, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dqf5EBY7KKMoxL3xrcmXZqujAArll', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec64c-5c93-7890-8e5e-8085f1face2b-0', tool_calls=[{'name': 're

## ویرایش

In [21]:
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "این دیگر از حد گذشت، اخراجی!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

{'email': 'سلام سینا، فردا برای جلسه\u200cمان دیر می\u200cرسم. می\u200cتوانیم '
          'وقت دیگری بگذاریم؟ با احترام، علیرضا.',
 'messages': [HumanMessage(content='لطفاً ایمیل من را بخوان و همین الان در همان thread پاسخ بده.', additional_kwargs={}, response_metadata={}, id='a4ddfffc-5068-4d58-a216-965844f6f3e0'),
              AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 403, 'prompt_tokens': 166, 'total_tokens': 569, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 384, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dqf5EBY7KKMoxL3xrcmXZqujAArll', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019ec64c-5c93-7890-8e5e-8085f1face2b-0', tool_calls=[{'name': 're